# Resources and Prompts

Tools _do_ things. **Resources** give the model data to read. **Prompts** are reusable templates the user can pick.


In [4]:
# !pip install -q "mcp>=2"

In [1]:
%%writefile server_rp.py
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("Notes")

NOTES = {
    "todo": "1. Record video\n2. Edit video\n3. Upload",
    "ideas": "Series on MCP, then knowledge graphs",
}


# :// -> its required in resource decorator
@mcp.resource("notes://{name}")
def get_note(name: str) -> str:
    """Read a note by name."""
    return NOTES.get(name, "Note not found")


@mcp.resource("notes://index")
def list_notes() -> str:
    """List all available notes."""
    return ", ".join(NOTES.keys())


@mcp.prompt()
def summarize_note(name: str) -> str:
    """Ask the model to summarize a note."""
    return f"Please read the note called '{name}' and summarize it in one sentence."


if __name__ == "__main__":
    mcp.run()

Overwriting server_rp.py


In [2]:
import os, sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def server_params(script):
    return StdioServerParameters(command=sys.executable, args=[script])


# Jupyter's stderr has no real file descriptor, so we send server logs to devnull
DEVNULL = open(os.devnull, "w")

In [3]:
async with stdio_client(server_params("server_rp.py"), errlog=DEVNULL) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        print("RESOURCES:")  # fixed resources
        for r in (await session.list_resources()).resources:
            print(" ", r.uri)

        print("RESOURCE TEMPLATES:")  # dynamic resources
        for t in (await session.list_resource_templates()).resource_templates:
            print(" ", t.uri_template)

        content = await session.read_resource("notes://todo")
        print("\nnotes://todo ->\n", content.contents[0].text)

        print("\nPROMPTS:")
        for p in (await session.list_prompts()).prompts:
            print(" ", p.name, "-", p.description)

        prompt = await session.get_prompt("summarize_note", {"name": "todo"})
        print("\nRendered prompt:", prompt.messages[0].content.text)

RESOURCES:
  notes://index
RESOURCE TEMPLATES:
  notes://{name}

notes://todo ->
 1. Record video
2. Edit video
3. Upload

PROMPTS:
  summarize_note - Ask the model to summarize a note.

Rendered prompt: Please read the note called 'todo' and summarize it in one sentence.


## When to use which?

| Primitive | Controlled by | Use for                         |
| --------- | ------------- | ------------------------------- |
| Tool      | the model     | actions, computation, API calls |
| Resource  | the app       | files, data, context            |
| Prompt    | the user      | reusable workflows              |

**Up next:** build a proper reusable MCP client.
